# Advanced Python `yield from`, Coroutines, and Generator Delegation
## Problems with complete solutions, tests, edge cases, and design notes

This notebook expands the supplied topic into a problem-driven advanced lesson.

### Learning objectives

By the end, you should be able to:

- explain exactly what `yield from` delegates;
- use `.send()`, `.throw()`, and `.close()` safely;
- capture a subgenerator's `return` value;
- design recursive and iterative lazy traversals;
- reason about generator state and delegation chains;
- build testable coroutine pipelines;
- handle cleanup, exceptions, cycles, and deep nesting;
- choose between generators, coroutines, iterators, and ordinary functions.

### Recommended workflow

1. Read each problem before opening the solution.
2. Predict every value returned by `next()` and `.send()`.
3. Run the tests.
4. Modify one assumption and add a new test.
5. Inspect generator state whenever behavior is surprising.

> Runtime target: Python 3.10 or newer.

## Core mental model

For:

```python
result = yield from subgenerator
```

the delegating generator forwards these operations to the subgenerator:

- `next(delegator)`
- `delegator.send(value)`
- `delegator.throw(...)`
- `delegator.close()`

When the subgenerator finishes with:

```python
return some_value
```

that value becomes the value of the `yield from` expression and is assigned to `result`.

A generator has four common states:

- `GEN_CREATED`
- `GEN_RUNNING`
- `GEN_SUSPENDED`
- `GEN_CLOSED`

In [1]:
from __future__ import annotations

from collections.abc import Generator, Iterable, Iterator, Mapping
from contextlib import contextmanager
from dataclasses import dataclass
from functools import wraps
import inspect
import math
from numbers import Real
from typing import Any, Callable, TypeVar

T = TypeVar("T")
R = TypeVar("R")

## Utility: consume a generator and capture its return value

A `for` loop hides the final `StopIteration.value`. This helper preserves it.

In [2]:
def exhaust_with_return(gen: Generator[T, Any, R]) -> tuple[list[T], R]:
    """Collect yielded items and the generator's final return value."""
    yielded: list[T] = []

    while True:
        try:
            yielded.append(next(gen))
        except StopIteration as exc:
            return yielded, exc.value

# Problem 1 — Build a safely primed request/response coroutine

Create a decorator that automatically primes a coroutine. Then implement a coroutine
that accepts values through `.send()` and yields the transformed result.

### Requirements

- The caller must not need to call `next()` manually.
- `.send(value)` must return the transformed value.
- Cleanup must still work with `.close()`.
- Preserve the wrapped function's metadata.

## Solution 1

A coroutine must reach its first `yield` before it can receive a non-`None` value.
The decorator performs that first `next()` exactly once.

In [3]:
def primed(
    coroutine_function: Callable[..., Generator[Any, Any, Any]]
) -> Callable[..., Generator[Any, Any, Any]]:
    """Decorator that creates and primes a coroutine."""
    @wraps(coroutine_function)
    def starter(*args: Any, **kwargs: Any) -> Generator[Any, Any, Any]:
        coroutine = coroutine_function(*args, **kwargs)
        next(coroutine)
        return coroutine

    return starter


@primed
def transformer(transform: Callable[[Any], Any]) -> Generator[Any, Any, None]:
    """Receive a value and yield transform(value)."""
    incoming = yield None

    while True:
        incoming = yield transform(incoming)

In [4]:
reverse = transformer(lambda value: value[::-1])

assert reverse.send("stressed") == "desserts"
assert reverse.send("drawer") == "reward"
assert inspect.getgeneratorstate(reverse) == "GEN_SUSPENDED"

reverse.close()
assert inspect.getgeneratorstate(reverse) == "GEN_CLOSED"

print("Problem 1 tests passed.")

Problem 1 tests passed.


### Best-practice notes

- Priming belongs at a clear API boundary; do not hide priming inconsistently.
- A decorator is helpful when every instance should start primed.
- A context manager is often better when deterministic cleanup is required.
- Do not send a non-`None` value to a generator in `GEN_CREATED` state.

# Problem 2 — Prove that `.send()` crosses a `yield from` boundary

Write a subgenerator that transforms incoming values. Wrap it with two layers of
delegation and verify that `.send()` reaches the deepest generator.

### Requirements

- Use two delegating generators.
- Return transformed values to the original caller.
- Add a validation rule in the deepest generator.

## Solution 2

In [5]:
def validating_transformer(
    transform: Callable[[str], str]
) -> Generator[str | None, str, None]:
    incoming = yield None

    while True:
        if not isinstance(incoming, str):
            raise TypeError("Only strings are accepted.")
        incoming = yield transform(incoming)


def middle_layer(
    transform: Callable[[str], str]
) -> Generator[str | None, str, None]:
    yield from validating_transformer(transform)


def outer_layer(
    transform: Callable[[str], str]
) -> Generator[str | None, str, None]:
    yield from middle_layer(transform)

In [6]:
delegator = outer_layer(str.upper)

assert next(delegator) is None
assert delegator.send("alpha") == "ALPHA"
assert delegator.send("Beta") == "BETA"

try:
    delegator.send(42)  # type: ignore[arg-type]
except TypeError as exc:
    assert str(exc) == "Only strings are accepted."
else:
    raise AssertionError("TypeError was expected.")

assert inspect.getgeneratorstate(delegator) == "GEN_CLOSED"
print("Problem 2 tests passed.")

Problem 2 tests passed.


### Reasoning

The original caller interacts only with `outer_layer`, but `yield from` establishes
a transparent communication channel through both delegators. The `TypeError`
originates in the deepest generator and propagates back to the caller.

# Problem 3 — Capture a subgenerator's `return` value

Implement a batch accumulator.

### Protocol

- Send numbers to add them to the current batch.
- Send `None` to finish the current batch.
- The subgenerator returns the batch subtotal.
- The delegator adds subtotals to a grand total.
- After each batch, the delegator yields a summary dictionary.
- Send `"stop"` after a summary to terminate and return the grand total.

## Solution 3

In [7]:
def collect_batch() -> Generator[None, float | None, float]:
    total = 0.0

    while True:
        value = yield

        if value is None:
            return total

        if not isinstance(value, Real):
            raise TypeError("Batch values must be real numbers.")

        total += float(value)


def batch_summer() -> Generator[dict[str, float] | None, float | None | str, float]:
    grand_total = 0.0

    while True:
        subtotal = yield from collect_batch()
        grand_total += subtotal

        command = yield {
            "subtotal": subtotal,
            "grand_total": grand_total,
        }

        if command == "stop":
            return grand_total

In [8]:
summer = batch_summer()

assert next(summer) is None
assert summer.send(10) is None
assert summer.send(20.5) is None
assert summer.send(None) == {"subtotal": 30.5, "grand_total": 30.5}

# This command resumes the delegator, which immediately enters and primes
# a fresh collect_batch() subgenerator.
assert summer.send("continue") is None
assert summer.send(4) is None
assert summer.send(6) is None
assert summer.send(None) == {"subtotal": 10.0, "grand_total": 40.5}

try:
    summer.send("stop")
except StopIteration as exc:
    assert exc.value == 40.5
else:
    raise AssertionError("The generator should have returned.")

print("Problem 3 tests passed.")

Problem 3 tests passed.


### Key observation

`return total` inside `collect_batch()` becomes `StopIteration(total)`.
`yield from` catches that internal `StopIteration` and evaluates to `total`.

# Problem 4 — Forward `.throw()` and recover from a custom exception

Create a running-average coroutine that supports a reset signal injected with
`.throw(ResetAverage)`.

### Requirements

- Sending numbers updates and yields the current average.
- Throwing `ResetAverage` clears the state without closing the coroutine.
- Sending `None` returns the final average.
- Place the coroutine behind a delegator.

## Solution 4

In [9]:
class ResetAverage(Exception):
    """Signal that the running average should be reset."""


def average_worker() -> Generator[float | None, float | None, float | None]:
    total = 0.0
    count = 0
    average: float | None = None

    while True:
        try:
            value = yield average
        except ResetAverage:
            total = 0.0
            count = 0
            average = None
            continue

        if value is None:
            return average

        if not isinstance(value, Real):
            raise TypeError("Average values must be real numbers.")

        total += float(value)
        count += 1
        average = total / count


def average_service() -> Generator[float | None, float | None, float | None]:
    result = yield from average_worker()
    return result

In [10]:
averager = average_service()

assert next(averager) is None
assert averager.send(10) == 10.0
assert averager.send(20) == 15.0

# .throw() crosses the yield-from boundary.
assert averager.throw(ResetAverage) is None
assert averager.send(7) == 7.0
assert averager.send(9) == 8.0

try:
    averager.send(None)
except StopIteration as exc:
    assert exc.value == 8.0
else:
    raise AssertionError("The generator should have returned.")

print("Problem 4 tests passed.")

Problem 4 tests passed.


### Best-practice notes

- Use custom exception types for control signals; avoid overloading ordinary errors.
- Catch only exceptions that the coroutine can meaningfully recover from.
- Let unexpected exceptions propagate.

# Problem 5 — Verify `.close()` propagation and deterministic cleanup

Build a resource-owning subgenerator and a delegator. Closing only the delegator
must execute both `finally` blocks.

## Solution 5

In [11]:
def resource_worker(events: list[tuple[str, Any]]) -> Generator[str, str, None]:
    try:
        message = yield "ready"

        while True:
            events.append(("received", message))
            message = yield "ready"
    finally:
        events.append(("worker", "closed"))


def resource_service(events: list[tuple[str, Any]]) -> Generator[str, str, None]:
    try:
        yield from resource_worker(events)
    finally:
        events.append(("service", "closed"))

In [12]:
events: list[tuple[str, Any]] = []
service = resource_service(events)

assert next(service) == "ready"
assert service.send("first") == "ready"
assert service.send("second") == "ready"

service.close()

assert events == [
    ("received", "first"),
    ("received", "second"),
    ("worker", "closed"),
    ("service", "closed"),
]
assert inspect.getgeneratorstate(service) == "GEN_CLOSED"

print("Problem 5 tests passed.")

Problem 5 tests passed.


### Design lesson

Put cleanup in `finally`, not after the main loop. Code after a suspended `yield`
may never run, but `finally` runs when `.close()` injects `GeneratorExit`.

# Problem 6 — Write a production-grade recursive lazy flattener

Flatten arbitrarily nested iterables using recursion and `yield from`.

### Requirements

- Treat strings, bytes, and bytearrays as atomic by default.
- Optionally descend into mappings.
- Support a maximum depth.
- Detect reference cycles.
- Stay lazy: do not construct the full flattened result internally.

## Solution 6

In [13]:
_ATOMIC_DEFAULT = (str, bytes, bytearray)


def flatten_recursive(
    value: Any,
    *,
    atomic_types: tuple[type, ...] = _ATOMIC_DEFAULT,
    descend_mappings: bool = False,
    max_depth: int | None = None,
    _depth: int = 0,
    _active_ids: set[int] | None = None,
) -> Iterator[Any]:
    """Lazily flatten nested iterables with cycle detection."""
    if max_depth is not None and max_depth < 0:
        raise ValueError("max_depth must be non-negative or None.")

    if isinstance(value, atomic_types):
        yield value
        return

    if max_depth is not None and _depth >= max_depth:
        yield value
        return

    if isinstance(value, Mapping):
        if not descend_mappings:
            yield value
            return
        iterable: Iterable[Any] = value.items()
    else:
        try:
            iterable = iter(value)
        except TypeError:
            yield value
            return

    if _active_ids is None:
        _active_ids = set()

    object_id = id(value)
    if object_id in _active_ids:
        raise ValueError("Cycle detected while flattening.")

    _active_ids.add(object_id)
    try:
        for item in iterable:
            yield from flatten_recursive(
                item,
                atomic_types=atomic_types,
                descend_mappings=descend_mappings,
                max_depth=max_depth,
                _depth=_depth + 1,
                _active_ids=_active_ids,
            )
    finally:
        _active_ids.remove(object_id)

In [14]:
nested = [
    1,
    ("abc", [2, 3]),
    {"x": [4, 5]},
    bytearray(b"z"),
]

assert list(flatten_recursive(nested)) == [
    1,
    "abc",
    2,
    3,
    {"x": [4, 5]},
    bytearray(b"z"),
]

assert list(
    flatten_recursive(nested, descend_mappings=True)
) == [
    1,
    "abc",
    2,
    3,
    "x",
    4,
    5,
    bytearray(b"z"),
]

assert list(flatten_recursive([1, [2, [3]]], max_depth=2)) == [1, 2, [3]]

cyclic: list[Any] = []
cyclic.append(cyclic)

try:
    list(flatten_recursive(cyclic))
except ValueError as exc:
    assert "Cycle detected" in str(exc)
else:
    raise AssertionError("A cycle should have been detected.")

print("Problem 6 tests passed.")

Problem 6 tests passed.


### Complexity

For an acyclic structure containing `n` visited elements:

- Time: `O(n)`
- Auxiliary memory: `O(d)`, where `d` is nesting depth
- Output memory: `O(1)` beyond yielded values, because the function is lazy

The Python call stack still limits extreme recursion depth. The next problem removes
that limitation.

# Problem 7 — Build a stack-safe iterative flattener

Reimplement flattening without recursive calls.

### Requirements

- Handle nesting deeper than Python's recursion limit.
- Preserve left-to-right traversal.
- Treat configurable atomic types as leaves.
- Detect cycles.

## Solution 7

In [15]:
def flatten_iterative(
    root: Any,
    *,
    atomic_types: tuple[type, ...] = _ATOMIC_DEFAULT,
) -> Iterator[Any]:
    """Stack-safe lazy flattening using explicit iterator frames."""
    # Each frame is (iterator, owner_id). owner_id is removed from the
    # active set when that iterator is exhausted.
    stack: list[tuple[Iterator[Any], int | None]] = [(iter((root,)), None)]
    active_ids: set[int] = set()

    while stack:
        iterator, owner_id = stack[-1]

        try:
            item = next(iterator)
        except StopIteration:
            stack.pop()
            if owner_id is not None:
                active_ids.remove(owner_id)
            continue

        if isinstance(item, atomic_types):
            yield item
            continue

        try:
            child_iterator = iter(item)
        except TypeError:
            yield item
            continue

        item_id = id(item)
        if item_id in active_ids:
            raise ValueError("Cycle detected while flattening.")

        active_ids.add(item_id)
        stack.append((child_iterator, item_id))

In [16]:
deep: Any = 0
for _ in range(5_000):
    deep = [deep]

assert list(flatten_iterative(deep)) == [0]
assert list(flatten_iterative([1, (2, [3, "abc"])])) == [1, 2, 3, "abc"]

cyclic_again: list[Any] = []
cyclic_again.append(cyclic_again)

try:
    list(flatten_iterative(cyclic_again))
except ValueError as exc:
    assert "Cycle detected" in str(exc)
else:
    raise AssertionError("A cycle should have been detected.")

print("Problem 7 tests passed.")

Problem 7 tests passed.


### Trade-off

The iterative version is more verbose but avoids recursion-depth failures.
The recursive `yield from` version is usually clearer for naturally recursive data.

# Problem 8 — Traverse a tree and return aggregate metadata

A generator can yield a stream and also return a summary.

Create a depth-first traversal that:

- yields node labels in pre-order;
- recursively delegates to each child;
- returns the total number of visited nodes.

## Solution 8

In [17]:
@dataclass(frozen=True)
class Node:
    label: str
    children: tuple["Node", ...] = ()


def walk_preorder(node: Node) -> Generator[str, None, int]:
    count = 1
    yield node.label

    for child in node.children:
        child_count = yield from walk_preorder(child)
        count += child_count

    return count

In [18]:
tree = Node(
    "root",
    (
        Node("left", (Node("left.left"), Node("left.right"))),
        Node("right", (Node("right.left"),)),
    ),
)

labels, node_count = exhaust_with_return(walk_preorder(tree))

assert labels == [
    "root",
    "left",
    "left.left",
    "left.right",
    "right",
    "right.left",
]
assert node_count == 6

print("Problem 8 tests passed.")

Problem 8 tests passed.


### Pattern

Each recursive call returns local metadata. The parent receives it through
`yield from`, combines it, and returns a larger summary.

# Problem 9 — Design a command-driven streaming statistics coroutine

Create a coroutine that receives numbers and yields immutable snapshots.

### Commands

- Send a number: update statistics.
- Send `RESET`: clear all statistics.
- Send `STOP`: return the last snapshot.

### Statistics

- count
- total
- minimum
- maximum
- mean

## Solution 9

In [19]:
RESET = object()
STOP = object()


@dataclass(frozen=True)
class Stats:
    count: int
    total: float
    minimum: float
    maximum: float
    mean: float


def statistics_worker() -> Generator[Stats | None, Real | object, Stats | None]:
    count = 0
    total = 0.0
    minimum = math.inf
    maximum = -math.inf
    snapshot: Stats | None = None

    while True:
        command = yield snapshot

        if command is STOP:
            return snapshot

        if command is RESET:
            count = 0
            total = 0.0
            minimum = math.inf
            maximum = -math.inf
            snapshot = None
            continue

        if not isinstance(command, Real):
            raise TypeError("Expected a real number, RESET, or STOP.")

        value = float(command)
        count += 1
        total += value
        minimum = min(minimum, value)
        maximum = max(maximum, value)

        snapshot = Stats(
            count=count,
            total=total,
            minimum=minimum,
            maximum=maximum,
            mean=total / count,
        )


def statistics_service() -> Generator[Stats | None, Real | object, Stats | None]:
    return (yield from statistics_worker())

In [20]:
stats = statistics_service()

assert next(stats) is None
assert stats.send(10) == Stats(1, 10.0, 10.0, 10.0, 10.0)
assert stats.send(20) == Stats(2, 30.0, 10.0, 20.0, 15.0)
assert stats.send(RESET) is None
assert stats.send(-4) == Stats(1, -4.0, -4.0, -4.0, -4.0)

try:
    stats.send(STOP)
except StopIteration as exc:
    assert exc.value == Stats(1, -4.0, -4.0, -4.0, -4.0)
else:
    raise AssertionError("The statistics service should have returned.")

print("Problem 9 tests passed.")

Problem 9 tests passed.


### Best-practice notes

- Use unique sentinel objects instead of magic strings when commands and data may overlap.
- Return immutable snapshots to prevent callers from corrupting internal state.
- Validate data at the boundary.

# Problem 10 — Build a coroutine pipeline with cleanup propagation

Create a three-stage pipeline:

1. normalize text;
2. keep only records satisfying a predicate;
3. collect accepted records.

Closing the outer pipeline must close every downstream stage.

## Solution 10

In [21]:
def collector(
    output: list[str],
    lifecycle: list[str],
) -> Generator[None, str, None]:
    try:
        while True:
            item = yield
            output.append(item)
    finally:
        lifecycle.append("collector closed")


def filter_stage(
    predicate: Callable[[str], bool],
    downstream: Generator[None, str, None],
    lifecycle: list[str],
) -> Generator[None, str, None]:
    next(downstream)

    try:
        while True:
            item = yield
            if predicate(item):
                downstream.send(item)
    finally:
        downstream.close()
        lifecycle.append("filter closed")


def normalize_stage(
    downstream: Generator[None, str, None],
    lifecycle: list[str],
) -> Generator[None, str, None]:
    next(downstream)

    try:
        while True:
            item = yield
            normalized = " ".join(str(item).strip().lower().split())
            downstream.send(normalized)
    finally:
        downstream.close()
        lifecycle.append("normalize closed")


def text_pipeline(
    output: list[str],
    lifecycle: list[str],
) -> Generator[None, str, None]:
    sink = collector(output, lifecycle)
    filtered = filter_stage(lambda text: len(text) >= 5, sink, lifecycle)
    yield from normalize_stage(filtered, lifecycle)

In [22]:
accepted: list[str] = []
lifecycle: list[str] = []

pipeline = text_pipeline(accepted, lifecycle)
next(pipeline)

pipeline.send("  ALPHA  ")
pipeline.send(" x ")
pipeline.send("Hello     World")
pipeline.send("four")

pipeline.close()

assert accepted == ["alpha", "hello world"]
assert lifecycle == [
    "collector closed",
    "filter closed",
    "normalize closed",
]

print("Problem 10 tests passed.")

Problem 10 tests passed.


### Architecture note

Only the outermost generator is exposed to the caller. Each stage owns and closes
the stage immediately downstream. This creates a simple ownership chain.

# Problem 11 — Manage coroutine lifetime with a context manager

A priming decorator starts a coroutine but does not guarantee closure.
Create a context manager that primes a generator on entry and closes it on exit,
even when the caller raises an exception.

## Solution 11

In [23]:
@contextmanager
def managed_coroutine(
    coroutine: Generator[Any, Any, Any]
) -> Iterator[Generator[Any, Any, Any]]:
    try:
        next(coroutine)
        yield coroutine
    finally:
        coroutine.close()

In [24]:
cleanup_log: list[str] = []


def logging_coroutine() -> Generator[None, str, None]:
    try:
        while True:
            message = yield
            cleanup_log.append(f"message:{message}")
    finally:
        cleanup_log.append("closed")


try:
    with managed_coroutine(logging_coroutine()) as logger:
        logger.send("one")
        logger.send("two")
        raise RuntimeError("simulated caller failure")
except RuntimeError:
    pass

assert cleanup_log == ["message:one", "message:two", "closed"]
print("Problem 11 tests passed.")

Problem 11 tests passed.


### Best-practice rule

Use a context manager when a coroutine owns resources or must flush state.
This makes lifetime management visible and exception-safe.

# Problem 12 — Inspect a live `yield from` delegation chain

Construct a three-level delegation chain and inspect `gi_yieldfrom`.

### Goal

Return the function names from the outermost generator down to the currently
active leaf generator.

## Solution 12

In [25]:
def leaf_coroutine() -> Generator[str, str, None]:
    item = yield "leaf-ready"
    while True:
        item = yield f"leaf:{item}"


def level_two() -> Generator[str, str, None]:
    yield from leaf_coroutine()


def level_one() -> Generator[str, str, None]:
    yield from level_two()


def delegation_chain(gen: Generator[Any, Any, Any]) -> list[str]:
    names: list[str] = []
    current: Any = gen

    while inspect.isgenerator(current):
        names.append(current.gi_code.co_name)
        current = current.gi_yieldfrom

    return names

In [26]:
chain_root = level_one()

assert inspect.getgeneratorstate(chain_root) == "GEN_CREATED"
assert next(chain_root) == "leaf-ready"
assert inspect.getgeneratorstate(chain_root) == "GEN_SUSPENDED"

assert delegation_chain(chain_root) == [
    "level_one",
    "level_two",
    "leaf_coroutine",
]

assert chain_root.send("hello") == "leaf:hello"
chain_root.close()

assert inspect.getgeneratorstate(chain_root) == "GEN_CLOSED"
print("Problem 12 tests passed.")

Problem 12 tests passed.


### Debugging value

`gi_yieldfrom` is useful when diagnosing which subgenerator currently owns control.
Do not build application logic around CPython-specific inspection details unless
you explicitly accept that implementation dependency.

# Problem 13 — Sequential workflow with per-step return values

Model a workflow where each task yields progress messages and returns a structured
summary. The parent delegates to tasks one at a time, collects their return values,
then returns the complete workflow report.

## Solution 13

In [27]:
@dataclass(frozen=True)
class TaskResult:
    name: str
    steps_completed: int


def task(name: str, steps: int) -> Generator[str, None, TaskResult]:
    if steps < 0:
        raise ValueError("steps must be non-negative.")

    for step_number in range(1, steps + 1):
        yield f"{name}: step {step_number}/{steps}"

    return TaskResult(name=name, steps_completed=steps)


def workflow(
    specification: Iterable[tuple[str, int]]
) -> Generator[str, None, list[TaskResult]]:
    results: list[TaskResult] = []

    for name, steps in specification:
        result = yield from task(name, steps)
        results.append(result)

    return results

In [28]:
progress, report = exhaust_with_return(
    workflow(
        [
            ("extract", 2),
            ("transform", 3),
            ("load", 1),
        ]
    )
)

assert progress == [
    "extract: step 1/2",
    "extract: step 2/2",
    "transform: step 1/3",
    "transform: step 2/3",
    "transform: step 3/3",
    "load: step 1/1",
]

assert report == [
    TaskResult("extract", 2),
    TaskResult("transform", 3),
    TaskResult("load", 1),
]

print("Problem 13 tests passed.")

Problem 13 tests passed.


### Generalization

This pattern works well for parsers, import pipelines, recursive searches, test
orchestration, and any workflow that emits progress while accumulating structured
results.

# Problem 14 — Recoverable batch ingestion with delegated validation

Build a batch-ingestion service.

### Protocol

- Send dictionaries representing records.
- Send `END_BATCH` to finish a batch.
- Valid records require a non-empty `"id"` and a numeric `"value"`.
- On a bad record, yield `("error", message)`.
- After an error, send `"continue"` to start a fresh batch or `"stop"` to return all
  previously committed records.
- On a valid batch, yield `("ok", committed_batch)`.
- After a valid batch, send `"continue"` or `"stop"`.

## Solution 14

In [29]:
END_BATCH = object()


class InvalidRecord(ValueError):
    """Raised when an input record fails validation."""


def validate_batch() -> Generator[None, dict[str, Any] | object, list[dict[str, Any]]]:
    accepted: list[dict[str, Any]] = []

    while True:
        record = yield

        if record is END_BATCH:
            return accepted

        if not isinstance(record, dict):
            raise InvalidRecord("Record must be a dictionary.")

        if not record.get("id"):
            raise InvalidRecord("Record requires a non-empty 'id'.")

        if not isinstance(record.get("value"), Real):
            raise InvalidRecord("Record requires a numeric 'value'.")

        # Copy on ingest so later caller mutation cannot alter committed data.
        accepted.append(dict(record))


def ingestion_service() -> Generator[
    tuple[str, Any] | None,
    dict[str, Any] | object | str,
    list[dict[str, Any]],
]:
    committed: list[dict[str, Any]] = []

    while True:
        try:
            batch = yield from validate_batch()
        except InvalidRecord as exc:
            command = yield ("error", str(exc))
        else:
            committed.extend(batch)
            command = yield ("ok", batch)

        if command == "stop":
            return committed

        if command != "continue":
            raise ValueError("Expected 'continue' or 'stop'.")

In [30]:
ingestor = ingestion_service()

assert next(ingestor) is None
assert ingestor.send({"id": "a", "value": 10}) is None
assert ingestor.send({"id": "b", "value": 20.5}) is None
assert ingestor.send(END_BATCH) == (
    "ok",
    [
        {"id": "a", "value": 10},
        {"id": "b", "value": 20.5},
    ],
)

assert ingestor.send("continue") is None
assert ingestor.send({"id": "", "value": 99}) == (
    "error",
    "Record requires a non-empty 'id'.",
)

# The failed batch is discarded; begin another batch.
assert ingestor.send("continue") is None
assert ingestor.send({"id": "c", "value": -1}) is None
assert ingestor.send(END_BATCH) == (
    "ok",
    [{"id": "c", "value": -1}],
)

try:
    ingestor.send("stop")
except StopIteration as exc:
    assert exc.value == [
        {"id": "a", "value": 10},
        {"id": "b", "value": 20.5},
        {"id": "c", "value": -1},
    ]
else:
    raise AssertionError("The ingestion service should have returned.")

print("Problem 14 tests passed.")

Problem 14 tests passed.


### Transactional interpretation

A batch is committed only after the validation subgenerator returns normally.
If validation raises, the partial batch remains local to the failed subgenerator and
is discarded when that generator closes.

# Problem 15 — Event router with multiple coroutine sinks

Create an event router that forwards each event to a sink selected by event type.

### Requirements

- Prime every sink exactly once.
- Ignore unknown event types.
- Close all unique sinks when the router closes.
- Expose the router through a `yield from` delegator.

## Solution 15

In [31]:
def event_collector(
    name: str,
    output: list[tuple[str, Any]],
) -> Generator[None, dict[str, Any], None]:
    try:
        while True:
            event = yield
            output.append((name, event["payload"]))
    finally:
        output.append((name, "closed"))


def event_router(
    routes: Mapping[str, Generator[None, dict[str, Any], None]]
) -> Generator[None, dict[str, Any], None]:
    unique_sinks = list(dict.fromkeys(routes.values()))

    for sink in unique_sinks:
        next(sink)

    try:
        while True:
            event = yield

            if not isinstance(event, dict):
                raise TypeError("Event must be a dictionary.")

            event_type = event.get("type")
            sink = routes.get(event_type)

            if sink is not None:
                sink.send(event)
    finally:
        for sink in unique_sinks:
            sink.close()


def event_application(
    routes: Mapping[str, Generator[None, dict[str, Any], None]]
) -> Generator[None, dict[str, Any], None]:
    yield from event_router(routes)

In [32]:
event_log: list[tuple[str, Any]] = []

audit_sink = event_collector("audit", event_log)
metric_sink = event_collector("metric", event_log)

app = event_application(
    {
        "login": audit_sink,
        "logout": audit_sink,
        "latency": metric_sink,
    }
)

next(app)
app.send({"type": "login", "payload": "user-1"})
app.send({"type": "latency", "payload": 23.4})
app.send({"type": "unknown", "payload": "ignored"})
app.send({"type": "logout", "payload": "user-1"})
app.close()

assert event_log == [
    ("audit", "user-1"),
    ("metric", 23.4),
    ("audit", "user-1"),
    ("audit", "closed"),
    ("metric", "closed"),
]

print("Problem 15 tests passed.")

Problem 15 tests passed.


# Challenge extensions with solution sketches

These are additional advanced modifications. Implement them in new cells.

## Challenge A — Async comparison

Rewrite one pipeline using `async def` and async generators.

**Solution sketch:** use `async for` for pull-based async streams. Note that async
generators do not support non-empty `return` values, so return summaries through
another channel or object.

## Challenge B — Backpressure

Make a sink accept at most `k` records before yielding a `"flush"` signal.

**Solution sketch:** keep a buffer in the sink. Every `k` items, yield the buffer or
invoke a flush callback. Clearly define whether `.send()` returns acknowledgments.

## Challenge C — Selective exception recovery

Extend the event router so a failing sink is quarantined while other sinks continue.

**Solution sketch:** catch exceptions around `sink.send(event)`, record the failed
sink, close it, remove every route pointing to it, and continue.

## Challenge D — Path-aware flattening

Yield `(path, leaf)` instead of only each leaf.

**Solution sketch:** pass a tuple path through recursive calls. Append sequence
indices or mapping keys at each level.

## Challenge E — Bounded-depth tree traversal

Stop descending after a chosen depth but still yield the boundary node.

**Solution sketch:** add a `depth` argument and recurse only while
`depth < max_depth`.

## Challenge F — Property-based tests

Generate random nested structures and compare recursive and iterative flatteners.

**Solution sketch:** use Hypothesis recursive strategies, exclude cyclic values, and
assert both implementations produce identical sequences.

# Final review checklist

Before shipping generator-based code, verify:

- **Priming:** Is the first `next()` explicit and correctly owned?
- **Protocol:** Are accepted `.send()` values and yielded responses documented?
- **Termination:** Is the generator's `return` value meaningful and tested?
- **Errors:** Which exceptions are recovered, transformed, or propagated?
- **Cleanup:** Does `.close()` reach every owned subgenerator or downstream stage?
- **Atomic values:** Are strings, bytes, mappings, and custom containers handled?
- **Cycles:** Can recursive data reference itself?
- **Depth:** Could recursion exceed Python's call-stack limit?
- **Laziness:** Are results streamed instead of unnecessarily materialized?
- **Mutation:** Are inputs copied when later caller mutation would be unsafe?
- **Inspection:** Are generator states tested at important boundaries?
- **Alternatives:** Would an ordinary function, iterator class, context manager,
  `asyncio`, or queue communicate intent more clearly?

# Summary

`yield from` is more than syntactic sugar for a loop. It delegates an entire
generator protocol:

- values yielded outward;
- values sent inward;
- exceptions thrown inward;
- closure and cleanup;
- the subgenerator's final return value.

The most robust designs make that protocol explicit, validate boundary inputs,
use `finally` for cleanup, test state transitions, and choose recursive or iterative
traversal based on depth and clarity requirements.